In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 13


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.15722244232893
Epoch 2/100, Loss: 1.8437543846666813
Epoch 3/100, Loss: 2.0141322165727615
Epoch 4/100, Loss: 1.7682823240756989
Epoch 5/100, Loss: 1.5489450842142105
Epoch 6/100, Loss: 1.7798344641923904
Epoch 7/100, Loss: 1.732184685766697
Epoch 8/100, Loss: 2.3717499002814293
Epoch 9/100, Loss: 1.7459747567772865
Epoch 10/100, Loss: 2.064598008990288
Epoch 11/100, Loss: 2.020048499107361
Epoch 12/100, Loss: 1.903767965734005
Epoch 13/100, Loss: 1.794828623533249
Epoch 14/100, Loss: 1.9508339688181877
Epoch 15/100, Loss: 1.694912888109684
Epoch 16/100, Loss: 1.9259826615452766
Epoch 17/100, Loss: 1.763438992202282
Epoch 18/100, Loss: 1.8426083996891975


Epoch 19/100, Loss: 1.8378755077719688
Epoch 20/100, Loss: 1.9632327035069466
Epoch 21/100, Loss: 2.0303224250674248
Epoch 22/100, Loss: 1.870932124555111
Epoch 23/100, Loss: 1.7793524414300919
Epoch 24/100, Loss: 1.83793106675148
Epoch 25/100, Loss: 1.6935250833630562
Epoch 26/100, Loss: 1.8317216038703918
Epoch 27/100, Loss: 1.8938947319984436
Epoch 28/100, Loss: 1.9179208874702454
Epoch 29/100, Loss: 1.8613305017352104
Epoch 30/100, Loss: 2.122263640165329
Epoch 31/100, Loss: 2.466999150812626
Epoch 32/100, Loss: 1.860762421041727
Epoch 33/100, Loss: 1.971873514354229
Epoch 34/100, Loss: 1.9695050418376923
Epoch 35/100, Loss: 1.78306046128273
Epoch 36/100, Loss: 1.771308571100235


Epoch 37/100, Loss: 1.8742894381284714
Epoch 38/100, Loss: 1.8047052696347237
Epoch 39/100, Loss: 2.042877621948719
Epoch 40/100, Loss: 2.0365184247493744
Epoch 41/100, Loss: 1.7202024534344673
Epoch 42/100, Loss: 2.127950929105282
Epoch 43/100, Loss: 1.8331144154071808
Epoch 44/100, Loss: 2.0152652114629745
Epoch 45/100, Loss: 1.9131561517715454
Epoch 46/100, Loss: 2.009749472141266
Epoch 47/100, Loss: 1.7520898655056953
Epoch 48/100, Loss: 1.6593854054808617
Epoch 49/100, Loss: 2.0622864812612534
Epoch 50/100, Loss: 2.3439106047153473
Epoch 51/100, Loss: 1.7407652735710144
Epoch 52/100, Loss: 1.8638389930129051
Epoch 53/100, Loss: 2.022243469953537


Epoch 54/100, Loss: 1.7654443196952343
Epoch 55/100, Loss: 1.897328607738018
Epoch 56/100, Loss: 1.8629970662295818
Epoch 57/100, Loss: 1.928843755275011
Epoch 58/100, Loss: 2.037298507988453
Epoch 59/100, Loss: 1.9787287339568138
Epoch 60/100, Loss: 1.918592095375061
Epoch 61/100, Loss: 1.8004879727959633
Epoch 62/100, Loss: 2.0182103887200356
Epoch 63/100, Loss: 1.8642936572432518
Epoch 64/100, Loss: 1.7809222154319286
Epoch 65/100, Loss: 1.7854408249258995
Epoch 66/100, Loss: 1.9235184080898762
Epoch 67/100, Loss: 1.9067076593637466
Epoch 68/100, Loss: 1.8732793256640434
Epoch 69/100, Loss: 2.064356245100498
Epoch 70/100, Loss: 1.7604866549372673
Epoch 71/100, Loss: 1.7297587916254997


Epoch 72/100, Loss: 2.016981415450573
Epoch 73/100, Loss: 1.6698984429240227
Epoch 74/100, Loss: 1.689736358821392
Epoch 75/100, Loss: 1.8197989016771317
Epoch 76/100, Loss: 1.6535751596093178
Epoch 77/100, Loss: 1.9575156345963478
Epoch 78/100, Loss: 1.6514952257275581
Epoch 79/100, Loss: 1.9078747667372227
Epoch 80/100, Loss: 1.817993126809597
Epoch 81/100, Loss: 1.8677964396774769
Epoch 82/100, Loss: 2.0318542048335075
Epoch 83/100, Loss: 2.4920917712152004
Epoch 84/100, Loss: 1.7267206907272339
Epoch 85/100, Loss: 1.9119891300797462
Epoch 86/100, Loss: 2.0576676577329636
Epoch 87/100, Loss: 1.8565235808491707
Epoch 88/100, Loss: 1.7383969277143478
Epoch 89/100, Loss: 1.8297028951346874
Epoch 90/100, Loss: 2.005117639899254


...

Epoch 16/100, Loss: 1.4146853536367416
Epoch 17/100, Loss: 1.488315187394619
Epoch 18/100, Loss: 1.5078218877315521
Epoch 19/100, Loss: 1.3462410159409046
Epoch 20/100, Loss: 1.4387605302035809
Epoch 21/100, Loss: 1.3736869134008884
Epoch 22/100, Loss: 1.335800725966692
Epoch 23/100, Loss: 1.307766955345869
Epoch 24/100, Loss: 1.4915286861360073
Epoch 25/100, Loss: 1.362340435385704
Epoch 26/100, Loss: 1.4452083855867386
Epoch 27/100, Loss: 1.3975752927362919
Epoch 28/100, Loss: 1.4160002805292606
Epoch 29/100, Loss: 1.3373816162347794
Epoch 30/100, Loss: 1.4243354983627796
Epoch 31/100, Loss: 1.4135990776121616
Epoch 32/100, Loss: 1.45135098695755


Epoch 33/100, Loss: 1.3793619349598885
Epoch 34/100, Loss: 1.4163848794996738
Epoch 35/100, Loss: 1.4260714687407017
Epoch 36/100, Loss: 1.4900545142591
Epoch 37/100, Loss: 1.3584292009472847
Epoch 38/100, Loss: 1.3838606141507626
Epoch 39/100, Loss: 1.2653522044420242
Epoch 40/100, Loss: 1.691996917128563
Epoch 41/100, Loss: 1.444214791059494
Epoch 42/100, Loss: 1.430315937846899
Epoch 43/100, Loss: 1.4379014261066914
Epoch 44/100, Loss: 1.4508894570171833
Epoch 45/100, Loss: 1.3315522074699402
Epoch 46/100, Loss: 1.3492168672382832
Epoch 47/100, Loss: 1.3864094279706478
Epoch 48/100, Loss: 1.41779350861907


Epoch 49/100, Loss: 1.5004885122179985
Epoch 50/100, Loss: 1.433227326720953
Epoch 51/100, Loss: 1.3995082639157772
Epoch 52/100, Loss: 1.3576568774878979
Epoch 53/100, Loss: 1.4937837906181812
Epoch 54/100, Loss: 1.3829207494854927
Epoch 55/100, Loss: 1.1888606809079647
Epoch 56/100, Loss: 1.3354027308523655
Epoch 57/100, Loss: 1.4816168285906315
Epoch 58/100, Loss: 1.5173801928758621
Epoch 59/100, Loss: 1.3608751520514488
Epoch 60/100, Loss: 1.400079533457756
Epoch 61/100, Loss: 1.3345458209514618
Epoch 62/100, Loss: 1.3981545455753803
Epoch 63/100, Loss: 1.320253361016512
Epoch 64/100, Loss: 1.262231420725584
Epoch 65/100, Loss: 1.4628572948276997
Epoch 66/100, Loss: 1.406105849891901


Epoch 67/100, Loss: 1.3538761399686337
Epoch 68/100, Loss: 1.4537771493196487
Epoch 69/100, Loss: 1.4382215440273285
Epoch 70/100, Loss: 1.4138579107820988
Epoch 71/100, Loss: 1.3158859312534332
Epoch 72/100, Loss: 1.3863249979913235
Epoch 73/100, Loss: 1.2215758264064789
Epoch 74/100, Loss: 1.34701843932271
Epoch 75/100, Loss: 1.4416738450527191
Epoch 76/100, Loss: 1.2885706052184105
Epoch 77/100, Loss: 1.5348763018846512
Epoch 78/100, Loss: 1.381028477102518
Epoch 79/100, Loss: 1.4212778769433498
Epoch 80/100, Loss: 1.3160904981195927
Epoch 81/100, Loss: 1.459360621869564
Epoch 82/100, Loss: 1.327060416340828
Epoch 83/100, Loss: 1.460123173892498
Epoch 84/100, Loss: 1.3367442265152931


Epoch 85/100, Loss: 1.4012139774858952
Epoch 86/100, Loss: 1.379746288061142
Epoch 87/100, Loss: 1.4289426319301128
Epoch 88/100, Loss: 1.414736807346344
Epoch 89/100, Loss: 1.3723962269723415
Epoch 90/100, Loss: 1.4809481874108315
Epoch 91/100, Loss: 1.3522675931453705
Epoch 92/100, Loss: 1.324257306754589
Epoch 93/100, Loss: 1.4780235849320889
Epoch 94/100, Loss: 1.4966986924409866
Epoch 95/100, Loss: 1.3238384649157524
Epoch 96/100, Loss: 1.3596154749393463
Epoch 97/100, Loss: 1.452028639614582
Epoch 98/100, Loss: 1.3362916335463524
Epoch 99/100, Loss: 1.4521928541362286
Epoch 100/100, Loss: 1.485485501587391
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5442
